# anndata - Annotated data

Please read through the following sections for a very brief overview if you are unfamiliar with the Python package [_anndata_](https://anndata.readthedocs.io/en/latest/). For an in-depth overview head over to the anndata documentation and tutorials.

## AnnData data structure

[_anndata_](https://anndata.readthedocs.io/en/latest/) allows for and builds the backbone for storing and manipulating scRNA-seq datasets as it is tabular data in a single object. Consider the following use case: you have the count matrix from a scRNA-seq experiment and associated metadata for the cells and genes. Using pandas, for example, you would work with three variables: one for the counts, one for the cell metadata, and one for the gene metadata. So when you filter out cells, you will have to update both the counts and cell metadata variables - with anndata, your variables are contained in a single variable - the AnnData object -, aligned along shared axes, and each subsetting updates all relevant entries automatically. Another major advantage of anndata is its support of sparse data structures.

The schematic below illustrates the general structure of a an AnnData object:

1. _X_ contains your cell by gene matrix
2. _layers_ contains additional matrices of dimension cell by gene such as a transformation of _X_
3. _obs_ is a pandas DataFrame collecting all cell-specific data
4. _var_ is a pandas DataFrame collecting all gene-specific data
5. _obsm_ and _varm_ contain matrices whose rows are aligned with the rows and colums of _X_, respectively, but the number of columns is not pre-defined
6. _obsp_ and _varp_ contain square matrices aligned with the rows and colums of _X_, respectively
7. _uns_ is a dictionary that can hold any information

<img src="https://raw.githubusercontent.com/scverse/anndata/main/docs/_static/img/anndata_schema.svg" alt="genescore" width="600">

## Library imports

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

import anndata as ad

## AnnData basics

Let's define a simple AnnData object, that collects a count matrix and some metadata and perform some basic manipulations.

In [2]:
n_obs = 10  # Number of cells
n_vars = 15  # Number of genes
adata = ad.AnnData(csr_matrix(np.random.poisson(1, size=(n_obs, n_vars)), dtype=np.float32))

# Define cell metadata
adata.obs = pd.DataFrame(
    {
        "patient_id": np.random.choice(["patient_1", "patient_2", "patient_3"], size=n_obs, replace=True),
        "site": np.random.choice(["MSK", "WCM"], size=n_obs, replace=True),
    },
    index=[f"obs_{obs_id}" for obs_id in range(n_obs)],
)
# Set data type of obs columns
adata.obs["patient_id"] = adata.obs["patient_id"].astype("category")

# Set names for variables (columns; genes)
adata.var_names = [f"var_{var_id}" for var_id in range(n_vars)]

adata

AnnData object with n_obs × n_vars = 10 × 15
    obs: 'patient_id', 'site'

In [3]:
# Access cell names
adata.obs_names

Index(['obs_0', 'obs_1', 'obs_2', 'obs_3', 'obs_4', 'obs_5', 'obs_6', 'obs_7',
       'obs_8', 'obs_9'],
      dtype='object')

In [4]:
# Access number of cells
adata.n_obs

10

In [5]:
# Access count data stored in X
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 95 stored elements and shape (10, 15)>

In [6]:
# Subset to patients 1 and 2
obs_mask = adata.obs["patient_id"].isin(["patient_1", "patient_2"])
adata = adata[obs_mask, :].copy()

In [7]:
# Verify subsetting
adata.obs["patient_id"].cat.categories

Index(['patient_1', 'patient_2'], dtype='object')

In [8]:
# Subset to first five observations and seven features
adata[:5, :7]

View of AnnData object with n_obs × n_vars = 5 × 7
    obs: 'patient_id', 'site'

In [9]:
# Add data to layers
adata.layers["counts"] = adata.X.copy()
adata.layers["counts_dense"] = adata.X.toarray().copy()
adata

AnnData object with n_obs × n_vars = 6 × 15
    obs: 'patient_id', 'site'
    layers: 'counts', 'counts_dense'

In [10]:
# Add gene metadata
adata.var["highly_variable"] = np.random.choice([True, False], size=adata.n_vars, replace=True)
adata

AnnData object with n_obs × n_vars = 6 × 15
    obs: 'patient_id', 'site'
    var: 'highly_variable'
    layers: 'counts', 'counts_dense'